# ML baseline full source

Полный код baseline-логики из `ml_baseline.py`.

Здесь находятся функции нормализации текста, извлечения признаков, построения retrieval-индекса, правил скоринга и вспомогательных ML-утилит.


## imports_and_constants


In [ ]:
#!/usr/bin/env python3


## Expr: Expr


In [ ]:
"""Local ML baseline for matching seller items to procurement lots.

The script intentionally uses lightweight local methods only:
- robust CSV loading;
- text normalization and characteristic cleanup;
- TF-IDF candidate retrieval;
- local OKPD-prefix classifier for boosting;
- service/repair penalties for goods-like seller items;
- aggregation from procurement rows to lots.
"""

from __future__ import annotations

import argparse

import hashlib

import json

import pickle

import re

import time

from collections import Counter

from pathlib import Path

from typing import Iterable

import numpy as np

import pandas as pd

from scipy.sparse import csr_matrix, hstack, load_npz, save_npz

from sklearn.decomposition import TruncatedSVD

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import SGDClassifier

PURCHASE_REQUIRED_COLUMNS = [
    "platform_brief",
    "publish_date",
    "platform_number",
    "lot_number",
    "pn_lot",
    "procedure_name",
    "lot_subject",
    "unit_name",
    "unit_okpd_code",
    "tags",
]

SELLER_REQUIRED_COLUMNS = ["id", "Категория", "Наименование", "Характеристики"]

SELLER_COLUMN_ALIASES = {
    "id": ("id", "ID", "Id"),
    "Категория": ("Категория", "Категория "),
    "Наименование": (
        "Наименование",
        "Наименование ",
        "Наименвоание",
        "Наименвоание ",
        "Наименование товара",
    ),
    "Характеристики": ("Характеристики", "Характеристики "),
}

SERVICE_OKPD_PREFIXES = (
    "33.12",
    "33.13",
    "33.14",
    "33.17",
    "45.20",
    "81.",
    "95.11",
)

HARD_SERVICE_TERMS = (
    "ремонт",
    "обслуживан",
    "заправк",
    "диагностик",
    "монтаж",
    "замена",
)

GENERIC_SERVICE_TERMS = (
    "услуг",
    "работ",
)

SUPPLY_TERMS = (
    "поставка",
    "закупка",
    "приобретение",
)

NOISY_CHARACTERISTIC_KEYS = (
    "вид товаров",
    "виды товаров",
    "страна происхождения",
    "вес",
    "масса",
    "габарит",
    "ширина",
    "высота",
    "глубина",
    "длина",
    "цвет",
    "количество в упаковке",
    "грузополучатель",
    "способ поставки",
)

STOPWORDS = {
    "для",
    "или",
    "при",
    "над",
    "под",
    "без",
    "тип",
    "вид",
    "товар",
    "товары",
    "услуга",
    "услуги",
    "поставка",
    "закупка",
    "оказание",
    "нужд",
    "гб",
    "шт",
    "черный",
    "белый",
    "серый",
    "красный",
    "синий",
    "зеленый",
}

ACCESSORY_LIKE_TYPES = {
    "network_accessory",
    "ups",
    "hdd",
    "ssd",
    "data_storage",
    "nas_storage",
    "memory_card",
    "usb_flash",
    "ram",
    "keyboard",
    "mouse",
    "keyboard_mouse_set",
    "power_bank",
}

PRODUCT_TYPE_RULES = [
    ("3d_filament", r"3d принтер|3д принтер|petg|abs pro|филамент|filament|пластик для 3d|пластик для 3д|\bpla\b"),
    ("printer_cartridge", r"тонер|драм картридж|картридж.*(струйн|лазер|canon|epson|hp|xerox|kyocera|pantum|brother|мфу|принтер|печатающ)"),
    ("dj_equipment", r"\bdj\b|диджей|dj контроллер|dj проигрывател|проигрывател.*dj"),
    ("binding_machine", r"брошюровщик|переплетчик|combbind|переплетн[а-я ]{0,20}(машин|устройств)|переплетная машина"),
    ("fax_device", r"\bфакс\b|факсимильн"),
    ("plotter_paper", r"бумаг[а-я ]{0,30}(плоттер|широкоформат)|плоттерн[а-я]* бумаг|широкоформатн[а-я]* бумаг"),
    ("thin_client", r"тонк[а-я]* клиент|неттоп"),
    ("camera_device", r"видеокамер|фотоловушк|camcorder|экшн камер|\bкамера\b|\bptz\b"),
    ("conference_system", r"видеоконференц|конференц[а-я \-]{0,20}(систем|связ|комплект)|конгресс[а-я \-]{0,20}систем|speakerphone|спикерфон"),
    ("presentation_holder", r"менюхолдер|демосистем|визитниц|ценникодержател|карман[а-я ]{0,20}информац|подставк[а-я ]{0,20}(информац|визит)|стойк[а-я ]{0,30}(информац|реклам)"),
    ("craft_punch", r"дырокол|вырубк|скрапбукинг"),
    ("car_audio", r"автомагнитол"),
    ("media_player", r"медиаплеер|мультимедийн[а-я]* приставк|cd проигрывател|dvd проигрывател|винилов[а-я]* проигрывател|музыкальн[а-я]* центр|apple tv"),
    ("smartphone", r"смартфон|мобильн[а-я]* телефон"),
    ("voip_phone", r"voip телефон|ip телефон|usb телефон|телефон.*sip|терминал[а-я]* ip телефони"),
    ("phone_device", r"аппарат[а-я]* телефон|проводн[а-я]* телефон|стационарн[а-я]* телефон|цифров[а-я ]{0,16}аппарат|\bтелефон\b|факсимильн|факс"),
    ("radio_device", r"радиостанц"),
    ("video_recorder", r"видеорегистратор|dash cam|радар детектор"),
    ("power_bank", r"аккумулятор внешний|внешний аккумулятор|портативн[а-я]* аккумулятор|power bank|пауэрбанк"),
    ("voltage_stabilizer", r"стабилизатор[а-я ]{0,40}напряж|стабилизатор[а-я ]{0,20}(однофазн|трехфазн)"),
    ("network_accessory", r"wi fi|wifi|bluetooth|точк[а-я ]{0,10}доступ|патч ?корд|трансивер|медиаконвертер|сетев[а-я]* карт|карт[а-я ]{0,16}сетев|сетев[а-я]* тестер|lan tst|волоконно оптическ|sfp|rj45|ethernet"),
    ("keyboard_mouse_set", r"клавиатур[а-я ]{0,40}мышь|мышь[а-я ]{0,40}клавиатур|комплект[а-я ]{0,40}клавиатур[а-я ]{0,40}мыш"),
    ("keyboard", r"клавиатур"),
    ("mouse", r"мышь компьютерн|компьютерная мышь|\bмышь\b"),
    ("projector", r"проектор"),
    ("interactive_display", r"интерактивн[а-я]* (доск|панел|стол)|сенсорн[а-я]* панел|(доск|панел|стол)[а-я ]{0,20}интерактив|демонстрационн[а-я]* систем|флипчарт|\bflip chart\b|\bflip\b"),
    ("antenna", r"антенн"),
    ("info_kiosk", r"информационн[а-я]* киоск|инфомат|\bкиоск\b"),
    ("flagpole", r"флагшток|флажн[а-я]* продукц"),
    ("insecticide", r"дезинсек|таракан|родентицид|насеком|ядохимикат|ловушк[а-я ]{0,24}(таракан|мурав)|средств[а-я ]{0,40}(мурав|насеком)"),
    ("food_glaze", r"кондитерск[а-я]* глазур|шоколадн[а-я]* глазур|пищев[а-я]* глазур"),
    ("glaze", r"глазур|ангоб|пигмент|гипс скульптур"),
    ("laptop", r"ноутбук|портативн[а-я]* компьютер"),
    ("tablet", r"планшет|\bipad\b"),
    ("monoblock", r"моноблок"),
    ("pc_case", r"корпус компьютерн|корпус[а-я]* системн[а-я]* блок"),
    ("system_unit", r"системн[а-я]* блок|рабоч[а-я]* станц|персональн[а-я]* электро вычислительн[а-я]* машина"),
    ("server", r"\bсервер\b|rack server|blade server"),
    ("memory_card", r"карт[аы]? памяти|microsd|sdhc|sdxc|compactflash"),
    ("usb_flash", r"usb флеш|флеш накопител|флешка|flash drive|usb накопител"),
    ("hdd", r"жестк[а-я]* диск|жестк[а-я]* диски|\bhdd\b|hard disk|\bsata\b|\bsas\b"),
    ("ssd", r"\bssd\b|твердотельн[а-я]* накопител|solid state"),
    ("nas_storage", r"сетев[а-я]* накопител|\bnas\b|систем[аы]? хранения данных"),
    ("data_storage", r"накопител[а-я]* данных|дисков[а-я]* накопител"),
    ("ram", r"оперативн[а-я]* память|модул[а-я]* оперативной памяти|\bddr[0-9]"),
    ("motherboard", r"материнск[а-я]* плат|motherboard"),
    ("gpu", r"видеокарт|графическ[а-я]* карт|\bgeforce\b|\bradeon\b"),
    ("cpu", r"процессор|\bcpu\b|\bxeon\b|intel core|amd ryzen"),
    ("psu", r"блок питания|источник питания"),
    ("ups", r"\bибп\b|источник бесперебойного питания|\bups\b"),
    ("monitor", r"монитор(?!инг)"),
    ("tv", r"телевизор|\btv\b"),
    ("headphones", r"наушник|гарнитур"),
    ("microphone", r"микрофон"),
    ("speaker", r"акустическ|колонк|сабвуфер|звуков[а-я]* аппаратура|усилител[а-я ]{0,24}(мощност|звука|аудио|акуст)"),
    ("switch", r"коммутатор|\bswitch\b|ethernet коммутатор"),
    ("router_modem", r"роутер|маршрутизатор|модем|\brouter\b|\bmodem\b"),
    ("printer_mfu", r"принтер|\bмфу\b|сканер"),
    ("software", r"программн[а-я]* обеспеч|лиценз|право на использование|\bsoftware\b|kaspersky|secret net"),
    ("soap", r"\bмыло\b|моющ[а-я]* средств|чистящ[а-я]* средств"),
    ("plasticine", r"пластилин|лепк"),
    ("disinfectant", r"дезинфицир"),
]

COMPATIBLE_TYPE_GROUPS = [
    {"hdd", "ssd", "usb_flash", "memory_card", "nas_storage", "data_storage"},
    {"laptop", "tablet", "monoblock", "system_unit", "thin_client", "server"},
    {"headphones", "microphone", "speaker", "media_player", "car_audio"},
    {"keyboard_mouse_set", "keyboard", "mouse"},
    {"switch", "router_modem", "network_accessory"},
    {"monitor", "tv", "projector", "interactive_display"},
    {"voip_phone", "smartphone", "phone_device", "fax_device"},
    {"conference_system", "camera_device", "microphone", "speaker", "phone_device"},
    {"camera_device", "video_recorder"},
]


## Assign: Assign


In [ ]:
PRODUCT_TYPE_SIGNAL_PATTERNS = {
    "car_audio": r"автомагнитол|головн[а-я]* устройст|автомобильн[а-я ]{0,20}(аудио|радио)|cd mp3 hu",
    "binding_machine": r"брошюров|переплет|combbind",
    "conference_system": r"видеоконференц|конференц|конгресс|speakerphone|спикерфон",
    "craft_punch": r"дырокол|вырубк|скрапбукинг",
    "fax_device": r"\bфакс\b|факсимильн",
    "printer_cartridge": r"картридж|тонер|печатающ|оргтехник|мфу|принтер",
    "usb_flash": r"флеш|usb|накопител",
    "memory_card": r"карт[аы]? памяти|microsd|sdhc|sdxc|compactflash",
    "laptop": r"ноутбук|портативн[а-я]* компьютер",
    "tablet": r"планшет",
    "system_unit": r"системн[а-я]* блок|рабоч[а-я]* станц|персональн[а-я]* компьютер|персональн[а-я]* электро вычислительн[а-я]* машина",
    "server": r"\bсервер\b|rack server|blade server",
    "smartphone": r"смартфон|мобильн[а-я]* телефон",
    "phone_device": r"аппарат[а-я]* телефон|проводн[а-я]* телефон|стационарн[а-я]* телефон|\bтелефон\b|факсимильн|факс",
    "radio_device": r"радиостанц",
    "video_recorder": r"видеорегистратор|dash cam|радар",
    "voltage_stabilizer": r"стабилизатор[а-я ]{0,40}напряж",
    "network_accessory": r"wi fi|wifi|bluetooth|точк[а-я ]{0,10}доступ|патч ?корд|трансивер|медиаконвертер|сетев[а-я]* карт|карт[а-я ]{0,16}сетев|сетев[а-я]* тестер|lan tst|волоконно оптическ|sfp|rj45|ethernet",
    "hdd": r"жестк[а-я]* диск|\bhdd\b|hard disk",
    "ssd": r"\bssd\b|твердотельн[а-я]* накопител|solid state",
    "nas_storage": r"сетев[а-я]* накопител|\bnas\b|систем[аы]? хранения данных",
    "data_storage": r"накопител[а-я]* данных|дисков[а-я]* накопител",
    "ram": r"оперативн[а-я]* память|модул[а-я]* памяти|\bddr[0-9]",
    "keyboard_mouse_set": r"клавиатур[а-я ]{0,40}мышь|мышь[а-я ]{0,40}клавиатур",
    "keyboard": r"клавиатур",
    "mouse": r"мышь",
    "camera_device": r"камера|видеокамер|ptz|фотоловушк|camcorder",
    "projector": r"проектор",
    "interactive_display": r"интерактивн[а-я]* (доск|панел|стол)|демонстрационн[а-я]* систем|флипчарт",
    "plotter_paper": r"плоттер|широкоформат|рулонн[а-я]* бумаг|inkjetbondpaper|bond paper",
    "presentation_holder": r"менюхолдер|демосистем|ценникодержател|карман|подставк",
    "thin_client": r"неттоп|тонк[а-я]* клиент",
    "flagpole": r"флагшток|флажн[а-я]* продукц",
    "insecticide": r"дезинсек|таракан|родентицид|насеком|ядохимикат|ловушк[а-я ]{0,24}(таракан|мурав)|средств[а-я ]{0,40}(мурав|насеком)",
    "glaze": r"глазур|ангоб|пигмент",
    "speaker": r"акустическ|колонк|сабвуфер|звуков[а-я]* аппаратура|усилител[а-я ]{0,24}(мощност|звука|аудио|акуст)",
    "media_player": r"медиаплеер|мультимедийн[а-я]* приставк|проигрывател|apple tv|музыкальн[а-я]* центр",
    "tv": r"телевизор|\btv\b",
}

def find_input_file(root: Path, prefix: str, suffix: str) -> Path:
    matches = sorted(
        p for p in root.iterdir() if p.is_file() and p.name.startswith(prefix) and p.name.endswith(suffix)
    )
    if not matches:
        raise FileNotFoundError(f"Не найден файл {prefix}*{suffix} в {root}")
    return matches[0]

def normalize_text(value: object) -> str:
    if pd.isna(value):
        return ""
    text = str(value).lower().replace("ё", "е")
    text = re.sub(r"[^0-9a-zа-я]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def normalize_category(value: object) -> str:
    text = normalize_text(value)
    text = re.sub(r"\b\w{1,2}\b", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def parse_characteristics(value: object) -> str:
    """Keep only relatively specific key/value characteristic tokens."""
    if pd.isna(value):
        return ""
    text = str(value).strip().strip('"')
    parts: list[str] = []
    for raw_part in text.split(";"):
        raw_part = raw_part.strip().strip('"')
        if not raw_part or ":" not in raw_part:
            continue
        key, val = raw_part.split(":", 1)
        key = key.strip().strip(":").lower().replace("ё", "е")
        val = val.strip().strip(":").strip()
        if not key or not val:
            continue
        if val.lower() in {"0", "0.0", "0.00000", "нет", "false", "nan"}:
            continue
        if any(key.startswith(noisy_key) for noisy_key in NOISY_CHARACTERISTIC_KEYS):
            continue
        if len(val) > 90:
            continue
        parts.append(f"{key} {val}")
        if len(parts) >= 30:
            break
    return normalize_text(" ".join(parts))

def okpd_prefix(value: object, level: int = 5) -> str:
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if not text or not re.match(r"^\d", text):
        return ""
    return text[:level]

def has_service_signal(text: str, okpd: str = "") -> bool:
    if okpd and any(okpd.startswith(prefix) for prefix in SERVICE_OKPD_PREFIXES):
        return True
    if any(term in text for term in HARD_SERVICE_TERMS):
        return True
    if any(term in text for term in GENERIC_SERVICE_TERMS) and not any(term in text for term in SUPPLY_TERMS):
        return True
    return False

def detect_product_type(text: object) -> str:
    normalized = normalize_text(text)
    if "бесперебойного питания" in normalized or re.search(r"\bибп\b|\bups\b", normalized):
        return "ups"
    for product_type, pattern in PRODUCT_TYPE_RULES:
        if re.search(pattern, normalized):
            return product_type
    return ""

def detect_product_type_fallback(*texts: object) -> str:
    for text in texts:
        product_type = detect_product_type(text)
        if product_type:
            return product_type
    return ""

def resolve_seller_product_type(name_text: object, category_text: object, chars_text: object) -> str:
    name_type = detect_product_type(name_text)
    category_type = detect_product_type(category_text)
    chars_type = detect_product_type(chars_text)

    if name_type and category_type:
        if name_type == category_type or are_compatible_types(name_type, category_type):
            return name_type
        if name_type in ACCESSORY_LIKE_TYPES and category_type not in ACCESSORY_LIKE_TYPES:
            return category_type
        if category_type in ACCESSORY_LIKE_TYPES and name_type not in ACCESSORY_LIKE_TYPES:
            return name_type
        return name_type
    if name_type:
        return name_type
    if category_type:
        return category_type
    return chars_type

def are_compatible_types(left: str, right: str) -> bool:
    if not left or not right:
        return False
    if left == right:
        return True
    return any(left in group and right in group for group in COMPATIBLE_TYPE_GROUPS)

def has_expected_type_signal(seller_type: str, purchase_type: str, purchase_text: str) -> bool:
    if not seller_type:
        return True
    if purchase_type and are_compatible_types(seller_type, purchase_type):
        return True
    if purchase_type and not are_compatible_types(seller_type, purchase_type):
        return False
    pattern = PRODUCT_TYPE_SIGNAL_PATTERNS.get(seller_type)
    if not pattern:
        return True
    return bool(re.search(pattern, purchase_text))

def type_adjustment(
    seller_type: str,
    purchase_type: str,
    exact_bonus: float,
    compatible_bonus: float,
    mismatch_penalty: float,
) -> tuple[float, str]:
    if not seller_type or not purchase_type:
        return 0.0, "unknown"
    if seller_type == purchase_type:
        return exact_bonus, "exact"
    if are_compatible_types(seller_type, purchase_type):
        return compatible_bonus, "compatible"
    return -mismatch_penalty, "mismatch"

def significant_tokens(text: str) -> set[str]:
    tokens = set(re.findall(r"[0-9a-zа-я]{3,}", text.lower().replace("ё", "е")))
    return {token for token in tokens if token not in STOPWORDS}

def topk_sparse_row(row: csr_matrix, k: int) -> list[tuple[int, float]]:
    data = row.data
    indices = row.indices
    if len(data) == 0:
        return []
    actual_k = min(k, len(data))
    part = np.argpartition(data, -actual_k)[-actual_k:]
    ordered = part[np.argsort(data[part])[::-1]]
    return [(int(indices[i]), float(data[i])) for i in ordered]

def confidence_label(score: float) -> str:
    if score >= 0.45:
        return "high"
    if score >= 0.25:
        return "medium"
    return "low"

def clipped_score_100(score: float) -> int:
    return int(round(max(0.0, min(1.0, score / 0.75)) * 100))

def is_suitable_item(score: float, best_score: float, threshold: float, relative_threshold: float) -> bool:
    if best_score <= 0:
        return False
    return score >= threshold and score >= best_score * relative_threshold

def read_purchases(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=";", encoding="utf-8-sig", dtype=str)
    df = df.drop(columns=[col for col in df.columns if col.startswith("Unnamed:")], errors="ignore")
    missing = [col for col in PURCHASE_REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"В файле закупок нет колонок: {missing}")
    return df[PURCHASE_REQUIRED_COLUMNS].copy()

def read_seller_matrix(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=";", encoding="utf-8-sig", dtype=str)
    normalized_to_actual = {re.sub(r"\s+", " ", str(col)).strip().lower(): col for col in df.columns}
    rename_map: dict[str, str] = {}
    for canonical, aliases in SELLER_COLUMN_ALIASES.items():
        for alias in aliases:
            key = re.sub(r"\s+", " ", alias).strip().lower()
            actual = normalized_to_actual.get(key)
            if actual is not None:
                rename_map[actual] = canonical
                break
    df = df.rename(columns=rename_map)
    missing = [col for col in SELLER_REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"В матрице селлера нет колонок: {missing}")
    return df[SELLER_REQUIRED_COLUMNS].copy()


## FunctionDef: sample_seller_by_category


In [ ]:
def sample_seller_by_category(seller: pd.DataFrame, per_category_limit: int) -> tuple[pd.DataFrame, dict]:
    report = {
        "enabled": bool(per_category_limit > 0),
        "per_category_limit": int(per_category_limit),
        "original_rows": int(len(seller)),
        "selected_rows": int(len(seller)),
        "original_unique_categories": int(seller["Категория"].map(normalize_category).nunique()),
        "selected_unique_categories": int(seller["Категория"].map(normalize_category).nunique()),
        "deduplicated_rows": int(len(seller)),
        "coverage_rate": 1.0,
    }
    if per_category_limit <= 0 or seller.empty:
        return seller.copy(), report

    tmp = seller.copy()
    tmp["_category_norm"] = tmp["Категория"].map(normalize_category).fillna("")
    tmp["_name_norm"] = tmp["Наименование"].map(normalize_text).fillna("")
    tmp["_chars_norm"] = tmp["Характеристики"].map(parse_characteristics).fillna("")
    tmp["_row_order"] = np.arange(len(tmp))
    tmp["_priority"] = tmp["_name_norm"].str.len() * 3 + tmp["_chars_norm"].str.len()
    tmp = tmp.drop_duplicates(subset=["_category_norm", "_name_norm", "_chars_norm"], keep="first")
    deduplicated_rows = int(len(tmp))
    tmp = tmp.sort_values(
        ["_category_norm", "_priority", "_row_order"],
        ascending=[True, False, True],
        kind="mergesort",
    )
    sampled = (
        tmp.groupby("_category_norm", group_keys=False, sort=False)
        .head(per_category_limit)
        .sort_values("_row_order", kind="mergesort")
        .drop(columns=["_category_norm", "_name_norm", "_chars_norm", "_row_order", "_priority"])
        .reset_index(drop=True)
    )
    original_unique_categories = max(1, report["original_unique_categories"])
    report.update(
        {
            "selected_rows": int(len(sampled)),
            "selected_unique_categories": int(sampled["Категория"].map(normalize_category).nunique()),
            "deduplicated_rows": deduplicated_rows,
            "coverage_rate": float(
                sampled["Категория"].map(normalize_category).nunique() / original_unique_categories
            ),
        }
    )
    return sampled, report

def build_quality_report(purchases: pd.DataFrame, seller: pd.DataFrame) -> dict:
    dt = pd.to_datetime(purchases["publish_date"], errors="coerce")
    purchase_codes = purchases["unit_okpd_code"].dropna().astype(str).str.strip()
    seller_cat_norm = seller["Категория"].map(normalize_category)
    return {
        "purchases": {
            "rows": int(len(purchases)),
            "unique_lots": int(purchases["pn_lot"].nunique()),
            "unique_platform_numbers": int(purchases["platform_number"].nunique()),
            "missing": {col: int(purchases[col].isna().sum()) for col in purchases.columns},
            "date_min": None if dt.isna().all() else str(dt.min().date()),
            "date_max": None if dt.isna().all() else str(dt.max().date()),
            "bad_dates": int(dt.isna().sum()),
            "okpd_unique": int(purchase_codes.nunique()),
            "top_lot_rows_share_top10": float(
                purchases["pn_lot"].value_counts().head(10).sum() / max(1, len(purchases))
            ),
        },
        "seller_matrix": {
            "rows": int(len(seller)),
            "unique_ids": int(seller["id"].nunique()),
            "unique_categories_raw": int(seller["Категория"].nunique()),
            "unique_categories_normalized": int(seller_cat_norm.nunique()),
            "missing": {col: int(seller[col].isna().sum()) for col in seller.columns},
        },
    }

def train_okpd_classifier(
    x_docs: csr_matrix,
    purchases: pd.DataFrame,
    min_class_count: int,
    max_iter: int,
) -> tuple[SGDClassifier | None, dict]:
    labels = purchases["unit_okpd_code"].map(okpd_prefix)
    valid = labels.ne("") & purchases["unit_name"].notna()
    class_counts = labels[valid].value_counts()
    keep_classes = set(class_counts[class_counts >= min_class_count].index)
    train_mask = valid & labels.isin(keep_classes)
    report = {
        "enabled": True,
        "train_rows": int(train_mask.sum()),
        "classes": int(len(keep_classes)),
        "min_class_count": int(min_class_count),
    }
    if train_mask.sum() == 0 or len(keep_classes) < 2:
        report["enabled"] = False
        report["reason"] = "not_enough_classes"
        return None, report
    clf = SGDClassifier(
        loss="modified_huber",
        alpha=1e-5,
        max_iter=max_iter,
        tol=1e-3,
        random_state=42,
        n_jobs=-1,
    )
    clf.fit(x_docs[train_mask.to_numpy()], labels[train_mask].to_numpy())
    return clf, report

def build_combined_text_index(
    purchase_text: pd.Series,
    purchase_item_text: pd.Series,
    seller_text: pd.Series,
    args: argparse.Namespace,
) -> tuple[csr_matrix, csr_matrix, csr_matrix, dict]:
    word_vectorizer = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=args.min_df,
        max_df=args.max_df,
        max_features=args.max_features,
        sublinear_tf=True,
        dtype=np.float32,
    )
    x_word_docs = word_vectorizer.fit_transform(purchase_text.to_numpy())
    x_word_item_docs = word_vectorizer.transform(purchase_item_text.to_numpy())
    q_word_docs = word_vectorizer.transform(seller_text.to_numpy())

    report = {
        "word_vocabulary_size": int(len(word_vectorizer.vocabulary_)),
        "char_enabled": False,
    }

    char_weight = float(args.char_weight)
    if char_weight <= 0 or args.char_max_features <= 0:
        return x_word_docs, x_word_item_docs, q_word_docs, report

    word_weight = max(0.0, 1.0 - char_weight)
    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(args.char_ngram_min, args.char_ngram_max),
        min_df=args.char_min_df,
        max_df=args.char_max_df,
        max_features=args.char_max_features,
        sublinear_tf=True,
        dtype=np.float32,
    )
    x_char_docs = char_vectorizer.fit_transform(purchase_text.to_numpy())
    x_char_item_docs = char_vectorizer.transform(purchase_item_text.to_numpy())
    q_char_docs = char_vectorizer.transform(seller_text.to_numpy())

    x_docs = hstack(
        [
            x_word_docs.multiply(word_weight),
            x_char_docs.multiply(char_weight),
        ],
        format="csr",
    )
    x_item_docs = hstack(
        [
            x_word_item_docs.multiply(word_weight),
            x_char_item_docs.multiply(char_weight),
        ],
        format="csr",
    )
    q_docs = hstack(
        [
            q_word_docs.multiply(word_weight),
            q_char_docs.multiply(char_weight),
        ],
        format="csr",
    )
    report.update(
        {
            "char_enabled": True,
            "char_vocabulary_size": int(len(char_vectorizer.vocabulary_)),
            "char_weight": float(char_weight),
            "word_weight": float(word_weight),
        }
    )
    return x_docs, x_item_docs, q_docs, report

def format_okpd_prediction(classes: np.ndarray, probabilities: np.ndarray, topn: int = 5) -> str:
    if probabilities.size == 0:
        return ""
    top = np.argsort(probabilities)[-topn:][::-1]
    return "; ".join(f"{str(classes[i])}:{float(probabilities[i]):.3f}" for i in top)

def compact_join(values: pd.Series, limit: int) -> str:
    cleaned = []
    for value in values.dropna().astype(str):
        value = re.sub(r"\s+", " ", value).strip()
        if value and value not in cleaned:
            cleaned.append(value)
        if len(cleaned) >= limit:
            break
    return " | ".join(cleaned)

def topk_dense_scores(values: np.ndarray, k: int) -> list[tuple[int, float]]:
    if values.size == 0 or k <= 0:
        return []
    actual_k = min(k, values.shape[0])
    part = np.argpartition(values, -actual_k)[-actual_k:]
    ordered = part[np.argsort(values[part])[::-1]]
    return [(int(idx), float(values[idx])) for idx in ordered]

def normalize_dense_rows(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0.0] = 1.0
    return (matrix / norms).astype(np.float32, copy=False)


## FunctionDef: fit_text_vectorizers


In [ ]:
def fit_text_vectorizers(documents: pd.Series, args: argparse.Namespace) -> tuple[TfidfVectorizer, TfidfVectorizer | None, csr_matrix, dict]:
    word_vectorizer = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=args.min_df,
        max_df=args.max_df,
        max_features=args.max_features,
        sublinear_tf=True,
        dtype=np.float32,
    )
    x_word_docs = word_vectorizer.fit_transform(documents.to_numpy())
    report = {
        "word_vocabulary_size": int(len(word_vectorizer.vocabulary_)),
        "char_enabled": False,
    }

    char_weight = float(args.char_weight)
    if char_weight <= 0 or args.char_max_features <= 0:
        return word_vectorizer, None, x_word_docs, report

    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(args.char_ngram_min, args.char_ngram_max),
        min_df=args.char_min_df,
        max_df=args.char_max_df,
        max_features=args.char_max_features,
        sublinear_tf=True,
        dtype=np.float32,
    )
    x_char_docs = char_vectorizer.fit_transform(documents.to_numpy())
    word_weight = max(0.0, 1.0 - char_weight)
    x_docs = hstack(
        [
            x_word_docs.multiply(word_weight),
            x_char_docs.multiply(char_weight),
        ],
        format="csr",
    )
    report.update(
        {
            "char_enabled": True,
            "char_vocabulary_size": int(len(char_vectorizer.vocabulary_)),
            "char_weight": float(char_weight),
            "word_weight": float(word_weight),
        }
    )
    return word_vectorizer, char_vectorizer, x_docs, report

def transform_with_vectorizers(
    documents: pd.Series,
    word_vectorizer: TfidfVectorizer,
    char_vectorizer: TfidfVectorizer | None,
    args: argparse.Namespace,
) -> csr_matrix:
    x_word_docs = word_vectorizer.transform(documents.to_numpy())
    if char_vectorizer is None:
        return x_word_docs
    char_weight = float(args.char_weight)
    word_weight = max(0.0, 1.0 - char_weight)
    x_char_docs = char_vectorizer.transform(documents.to_numpy())
    return hstack(
        [
            x_word_docs.multiply(word_weight),
            x_char_docs.multiply(char_weight),
        ],
        format="csr",
    )

def build_lot_catalog(
    purchases: pd.DataFrame,
    purchase_item_base: pd.Series,
    purchase_tags: pd.Series,
    purchase_procedure: pd.Series,
    purchase_okpd_prefix: np.ndarray,
    lot_items_limit: int,
    lot_tags_limit: int,
) -> pd.DataFrame:
    temp = purchases[
        [
            "pn_lot",
            "platform_brief",
            "publish_date",
            "platform_number",
            "lot_number",
            "procedure_name",
            "lot_subject",
        ]
    ].copy()
    temp["item_display"] = [
        display_text(unit_name, lot_subject)
        for unit_name, lot_subject in zip(purchases["unit_name"].tolist(), purchases["lot_subject"].tolist())
    ]
    temp["item_base"] = purchase_item_base.to_numpy()
    temp["tags_norm"] = purchase_tags.to_numpy()
    temp["okpd_prefix"] = purchase_okpd_prefix

    lot_catalog = (
        temp.groupby("pn_lot", sort=False)
        .agg(
            platform_brief=("platform_brief", "first"),
            publish_date=("publish_date", "first"),
            platform_number=("platform_number", "first"),
            lot_number=("lot_number", "first"),
            procedure_name=("procedure_name", "first"),
            lot_subject=("lot_subject", "first"),
            item_display_sample=("item_display", lambda s: compact_join(pd.Series(s), lot_items_limit)),
            item_text_sample=("item_base", lambda s: compact_join(pd.Series(s), lot_items_limit)),
            tags_sample=("tags_norm", lambda s: compact_join(pd.Series(s), lot_tags_limit)),
            okpd_sample=("okpd_prefix", lambda s: compact_join(pd.Series(s), 8)),
            lot_row_count=("item_display", "size"),
        )
        .reset_index()
    )
    lot_subject_norm = lot_catalog["lot_subject"].map(normalize_text).fillna("")
    lot_items_norm = lot_catalog["item_text_sample"].map(normalize_text).fillna("")
    lot_tags_norm = lot_catalog["tags_sample"].map(normalize_text).fillna("")
    lot_procedure_norm = lot_catalog["procedure_name"].map(normalize_text).fillna("")
    lot_catalog["lot_text"] = (
        lot_subject_norm
        + " "
        + lot_subject_norm
        + " "
        + lot_items_norm
        + " "
        + lot_items_norm
        + " "
        + lot_tags_norm
        + " "
        + lot_procedure_norm
    ).str.strip()
    return lot_catalog

def build_index_cache_key(purchases_path: Path, args: argparse.Namespace) -> str:
    stat = purchases_path.stat()
    payload = {
        "version": 2,
        "path": str(purchases_path.resolve()),
        "mtime_ns": stat.st_mtime_ns,
        "size": stat.st_size,
        "max_features": args.max_features,
        "min_df": args.min_df,
        "max_df": args.max_df,
        "char_weight": args.char_weight,
        "char_max_features": args.char_max_features,
        "char_min_df": args.char_min_df,
        "char_max_df": args.char_max_df,
        "char_ngram_min": args.char_ngram_min,
        "char_ngram_max": args.char_ngram_max,
        "embedding_dim": args.embedding_dim,
        "embedding_random_state": args.embedding_random_state,
        "lot_items_limit": args.lot_items_limit,
        "lot_tags_limit": args.lot_tags_limit,
    }
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True).encode("utf-8")
    return hashlib.md5(raw).hexdigest()[:16]


## FunctionDef: build_or_load_hybrid_indices


In [ ]:
def build_or_load_hybrid_indices(
    root: Path,
    purchases_path: Path,
    lot_text: pd.Series,
    purchase_item_text: pd.Series,
    args: argparse.Namespace,
) -> tuple[TfidfVectorizer, TfidfVectorizer | None, csr_matrix, csr_matrix, TruncatedSVD | None, np.ndarray | None, dict]:
    cache_key = build_index_cache_key(purchases_path, args)
    cache_root = Path(args.index_cache_dir)
    if not cache_root.is_absolute():
        cache_root = root / cache_root
    cache_dir = cache_root / cache_key
    cache_dir.mkdir(parents=True, exist_ok=True)

    cache_files = {
        "word_vectorizer": cache_dir / "word_vectorizer.pkl",
        "char_vectorizer": cache_dir / "char_vectorizer.pkl",
        "lot_docs": cache_dir / "lot_docs.npz",
        "item_docs": cache_dir / "item_docs.npz",
        "svd": cache_dir / "svd.pkl",
        "lot_embeddings": cache_dir / "lot_embeddings.npy",
        "report": cache_dir / "report.json",
    }

    cache_ready = (
        not args.rebuild_index_cache
        and cache_files["word_vectorizer"].exists()
        and cache_files["lot_docs"].exists()
        and cache_files["item_docs"].exists()
        and cache_files["report"].exists()
        and (
            args.embedding_dim <= 0
            or (
                cache_files["svd"].exists()
                and cache_files["lot_embeddings"].exists()
            )
        )
    )

    if cache_ready:
        with cache_files["word_vectorizer"].open("rb") as fh:
            word_vectorizer = pickle.load(fh)
        char_vectorizer = None
        if cache_files["char_vectorizer"].exists():
            with cache_files["char_vectorizer"].open("rb") as fh:
                char_vectorizer = pickle.load(fh)
        x_lot_docs = load_npz(cache_files["lot_docs"]).tocsr()
        x_item_docs = load_npz(cache_files["item_docs"]).tocsr()
        lot_embeddings = None
        svd = None
        if args.embedding_dim > 0:
            with cache_files["svd"].open("rb") as fh:
                svd = pickle.load(fh)
            lot_embeddings = np.load(cache_files["lot_embeddings"])
        report = json.loads(cache_files["report"].read_text(encoding="utf-8"))
        report["cache_hit"] = True
        report["cache_dir"] = str(cache_dir)
        return word_vectorizer, char_vectorizer, x_lot_docs, x_item_docs, svd, lot_embeddings, report

    word_vectorizer, char_vectorizer, x_lot_docs, report = fit_text_vectorizers(lot_text, args)
    x_item_docs = transform_with_vectorizers(purchase_item_text, word_vectorizer, char_vectorizer, args)

    svd = None
    lot_embeddings = None
    if args.embedding_dim > 0:
        embedding_dim = min(
            int(args.embedding_dim),
            max(2, x_lot_docs.shape[1] - 1),
        )
        svd = TruncatedSVD(n_components=embedding_dim, random_state=args.embedding_random_state)
        lot_embeddings = normalize_dense_rows(svd.fit_transform(x_lot_docs).astype(np.float32, copy=False))
        report["embedding_enabled"] = True
        report["embedding_dim"] = int(embedding_dim)
    else:
        report["embedding_enabled"] = False
        report["embedding_dim"] = 0

    with cache_files["word_vectorizer"].open("wb") as fh:
        pickle.dump(word_vectorizer, fh)
    if char_vectorizer is not None:
        with cache_files["char_vectorizer"].open("wb") as fh:
            pickle.dump(char_vectorizer, fh)
    save_npz(cache_files["lot_docs"], x_lot_docs)
    save_npz(cache_files["item_docs"], x_item_docs)
    if svd is not None and lot_embeddings is not None:
        with cache_files["svd"].open("wb") as fh:
            pickle.dump(svd, fh)
        np.save(cache_files["lot_embeddings"], lot_embeddings)
    report["cache_hit"] = False
    report["cache_dir"] = str(cache_dir)
    cache_files["report"].write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    return word_vectorizer, char_vectorizer, x_lot_docs, x_item_docs, svd, lot_embeddings, report

def display_text(value: object, fallback: object = "") -> str:
    if pd.isna(value) or not str(value).strip():
        return "" if pd.isna(fallback) else re.sub(r"\s+", " ", str(fallback)).strip()
    return re.sub(r"\s+", " ", str(value)).strip()

def build_suspicion_flags(row: pd.Series, repeated_lot_ids: set[str]) -> list[str]:
    flags: list[str] = []
    seller_text = f"{row['товар_селлера']} {row['категория_селлера']}".lower()
    proposal_text = f"{row['предмет_лота']} {row['лучшая_позиция_в_лоте']}".lower()
    seller_type = str(row.get("тип_товара_селлера", "") or "")
    proposal_type = str(row.get("тип_предложения", "") or "")

    if row["уверенность"] == "low":
        flags.append("низкая уверенность")
    if int(row["подходящих_позиций_в_лоте"]) == 0:
        flags.append("нет подходящих позиций внутри лота")

    service_terms = r"ремонт|обслуживан|заправк|диагностик|монтаж|замена"
    seller_is_service = bool(re.search(service_terms, seller_text))
    proposal_is_service = bool(re.search(service_terms, proposal_text))
    if proposal_is_service and not seller_is_service:
        flags.append("товару предложен сервисный/ремонтный лот")
    if seller_type and proposal_type and not are_compatible_types(seller_type, proposal_type):
        flags.append(f"несовпадение типа: {seller_type} -> {proposal_type}")

    keyword_rules = [
        ("картридж", r"картридж|тонер|печатающ|оргтехник|мфу|принтер"),
        ("коммутатор", r"коммутатор|сетев"),
        ("телевизор", r"телевизор|электробытов|бытовая электрон"),
        ("наушник", r"наушник|гарнитур|аудио|звуков"),
        ("материнская плата", r"материнская плата|комплектующ"),
        ("флеш", r"флеш|usb|накопител|карт[аы]? памяти|microsd|sdhc|sdxc"),
        ("ноутбук", r"ноутбук|портативн[а-я]* компьютер"),
        ("планшет", r"планшет"),
        ("смартфон", r"смартфон|мобильн[а-я]* телефон"),
        ("автомагнитол", r"автомагнитол|автомобильн[а-я]* аудио|головн[а-я]* устройст"),
        ("стабилизатор напряжения", r"стабилизатор[а-я ]{0,40}напряж"),
    ]
    for seller_keyword, proposal_regex in keyword_rules:
        if seller_keyword in seller_text and not re.search(proposal_regex, proposal_text):
            flags.append(f"нет ожидаемого ключевого слова: {seller_keyword}")
    return flags

def map_problem_type(flag: str) -> str:
    flag = str(flag).strip()
    if flag == "низкая уверенность":
        return "low_confidence"
    if flag == "нет подходящих позиций внутри лота":
        return "no_suitable_item_in_best_lot"
    if flag == "товару предложен сервисный/ремонтный лот":
        return "service_lot_for_goods"
    if flag.startswith("несовпадение типа:"):
        return "type_mismatch"
    if flag.startswith("нет ожидаемого ключевого слова:"):
        return "missing_expected_keyword"
    return "other_issue"

def build_issue_outputs(review: pd.DataFrame, suspicious: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    issue_columns = [
        "problem_type",
        "товар_селлера",
        "категория_селлера",
        "id_товара",
        "тип_товара_селлера",
        "предлагаемый_лот",
        "предмет_лота",
        "лучшая_позиция_в_лоте",
        "тип_предложения",
        "тип_совпал",
        "релевантность_0_100",
        "уверенность",
        "подходящих_позиций_в_лоте",
        "позиция_взята_из_предмета_лота",
    ]
    if review.empty:
        empty_issue_audit = pd.DataFrame(columns=issue_columns)
        empty_issue_summary = pd.DataFrame(columns=["problem_type", "rows"])
        empty_category_issue_summary = pd.DataFrame(columns=["категория_селлера", "problem_type", "rows"])
        return empty_issue_audit, empty_issue_summary, empty_category_issue_summary

    issue_rows: list[dict] = []
    seen: set[tuple[str, str, str]] = set()

    def append_issue(problem_type: str, row: pd.Series) -> None:
        key = (str(row.get("id_товара", "")), str(row.get("предлагаемый_лот", "")), problem_type)
        if key in seen:
            return
        seen.add(key)
        issue_rows.append({column: row.get(column, "") for column in issue_columns if column != "problem_type"} | {"problem_type": problem_type})

    for _, row in review.iterrows():
        if bool(row["позиция_взята_из_предмета_лота"]):
            append_issue("lot_level_position_only", row)
        seller_type = row["тип_товара_селлера"]
        proposal_type = row["тип_предложения"]
        if pd.isna(seller_type) or not str(seller_type).strip():
            append_issue("seller_type_unknown", row)
        if pd.isna(proposal_type) or not str(proposal_type).strip():
            append_issue("proposal_type_unknown", row)

    if not suspicious.empty:
        for _, row in suspicious.iterrows():
            for raw_flag in str(row["флаги_проверки"]).split(";"):
                raw_flag = raw_flag.strip()
                if raw_flag:
                    append_issue(map_problem_type(raw_flag), row)

    issue_audit = pd.DataFrame(issue_rows, columns=issue_columns).sort_values(
        ["problem_type", "релевантность_0_100"],
        ascending=[True, True],
        kind="mergesort",
    )
    issue_summary = (
        issue_audit["problem_type"]
        .value_counts()
        .rename_axis("problem_type")
        .reset_index(name="rows")
    )
    category_issue_summary = (
        issue_audit.groupby(["категория_селлера", "problem_type"], dropna=False)
        .size()
        .reset_index(name="rows")
        .sort_values(["rows", "категория_селлера", "problem_type"], ascending=[False, True, True], kind="mergesort")
    )
    return issue_audit, issue_summary, category_issue_summary


## FunctionDef: build_category_metrics


In [ ]:
def build_category_metrics(review: pd.DataFrame, probable_errors: pd.DataFrame) -> pd.DataFrame:
    columns = [
        "категория_селлера",
        "категория_нормализованная",
        "rows",
        "high",
        "med_or_high",
        "low",
        "no_suitable",
        "lot_level_only",
        "known_type",
        "probable_errors",
        "mean_score",
        "med_or_high_rate",
        "low_rate",
        "no_suitable_rate",
        "lot_level_only_rate",
        "known_type_rate",
        "probable_error_rate",
    ]
    if review.empty:
        return pd.DataFrame(columns=columns)

    base = review.copy()
    base["категория_нормализованная"] = base["категория_селлера"].map(normalize_category)
    base["_score"] = pd.to_numeric(base["релевантность_0_100"], errors="coerce").fillna(0.0)
    base["_high"] = base["уверенность"].eq("high").astype(int)
    base["_med_or_high"] = base["уверенность"].isin(["medium", "high"]).astype(int)
    base["_low"] = base["уверенность"].eq("low").astype(int)
    base["_no_suitable"] = pd.to_numeric(base["подходящих_позиций_в_лоте"], errors="coerce").fillna(0).eq(0).astype(int)
    base["_lot_level_only"] = base["позиция_взята_из_предмета_лота"].astype(bool).astype(int)
    base["_known_type"] = base["тип_товара_селлера"].fillna("").astype(str).str.strip().ne("").astype(int)

    category_metrics = (
        base.groupby(["категория_селлера", "категория_нормализованная"], dropna=False)
        .agg(
            rows=("товар_селлера", "size"),
            high=("_high", "sum"),
            med_or_high=("_med_or_high", "sum"),
            low=("_low", "sum"),
            no_suitable=("_no_suitable", "sum"),
            lot_level_only=("_lot_level_only", "sum"),
            known_type=("_known_type", "sum"),
            mean_score=("_score", "mean"),
        )
        .reset_index()
    )
    category_metrics["med_or_high_rate"] = category_metrics["med_or_high"] / category_metrics["rows"].clip(lower=1)
    category_metrics["low_rate"] = category_metrics["low"] / category_metrics["rows"].clip(lower=1)
    category_metrics["no_suitable_rate"] = category_metrics["no_suitable"] / category_metrics["rows"].clip(lower=1)
    category_metrics["lot_level_only_rate"] = category_metrics["lot_level_only"] / category_metrics["rows"].clip(lower=1)
    category_metrics["known_type_rate"] = category_metrics["known_type"] / category_metrics["rows"].clip(lower=1)

    probable_by_category = (
        probable_errors.groupby("категория_селлера", dropna=False)
        .size()
        .rename("probable_errors")
        .reset_index()
        if not probable_errors.empty
        else pd.DataFrame(columns=["категория_селлера", "probable_errors"])
    )
    category_metrics = category_metrics.merge(probable_by_category, on="категория_селлера", how="left")
    category_metrics["probable_errors"] = category_metrics["probable_errors"].fillna(0).astype(int)
    category_metrics["probable_error_rate"] = category_metrics["probable_errors"] / category_metrics["rows"].clip(lower=1)
    return category_metrics[columns].sort_values(
        ["probable_error_rate", "low_rate", "rows", "категория_селлера"],
        ascending=[False, False, False, True],
        kind="mergesort",
    )


## FunctionDef: run


In [ ]:
def run(args: argparse.Namespace) -> None:
    started_at = time.time()
    root = Path(args.cwd).resolve()
    purchases_path = Path(args.purchases) if args.purchases else find_input_file(root, "Закупки", ".csv")
    seller_path = Path(args.seller) if args.seller else find_input_file(root, "Пример", ".csv")
    out_dir = Path(args.output_dir)
    if not out_dir.is_absolute():
        out_dir = root / out_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"[1/6] Loading data")
    purchases = read_purchases(purchases_path)
    seller_full = read_seller_matrix(seller_path)
    quality_report = build_quality_report(purchases, seller_full)
    seller, seller_sampling_report = sample_seller_by_category(seller_full, args.per_category_limit)

    print(f"[2/6] Preparing texts")
    purchase_unit = purchases["unit_name"].map(normalize_text).fillna("")
    purchase_tags = purchases["tags"].map(normalize_text).fillna("")
    purchase_subject = purchases["lot_subject"].map(normalize_text).fillna("")
    purchase_procedure = purchases["procedure_name"].map(normalize_text).fillna("")
    purchase_item_base = purchase_unit.where(purchase_unit.ne(""), purchase_subject)
    purchase_item_text = purchase_item_base + " " + purchase_item_base + " " + purchase_item_base
    purchase_text = purchase_item_base + " " + purchase_subject + " " + purchase_tags + " " + purchase_procedure
    purchase_type_signal_text = (purchase_item_base + " " + purchase_subject).tolist()
    purchase_okpd_prefix = purchases["unit_okpd_code"].map(okpd_prefix).fillna("").to_numpy()
    lot_catalog = build_lot_catalog(
        purchases=purchases,
        purchase_item_base=purchase_item_base,
        purchase_tags=purchase_tags,
        purchase_procedure=purchase_procedure,
        purchase_okpd_prefix=purchase_okpd_prefix,
        lot_items_limit=args.lot_items_limit,
        lot_tags_limit=args.lot_tags_limit,
    )
    lot_text = lot_catalog["lot_text"].fillna("")
    lot_ids = lot_catalog["pn_lot"].astype(str).tolist()

    seller_name = seller["Наименование"].map(normalize_text).fillna("")
    seller_category = seller["Категория"].map(normalize_category).fillna("")
    seller_chars = seller["Характеристики"].map(parse_characteristics).fillna("")
    seller_text = seller_name + " " + seller_name + " " + seller_name + " " + seller_category + " " + seller_chars
    seller_product_type = np.array(
        [
            resolve_seller_product_type(name, category, chars)
            for name, chars, category in zip(seller_name.tolist(), seller_chars.tolist(), seller_category.tolist())
        ],
        dtype=object,
    )
    purchase_product_type = np.array(
        [
            detect_product_type_fallback(item_base, subject)
            for item_base, subject in zip(purchase_item_base.tolist(), purchase_subject.tolist())
        ],
        dtype=object,
    )

    purchase_service = [
        has_service_signal(text, okpd_prefix(code))
        for text, code in zip(
            (purchase_procedure + " " + purchase_subject + " " + purchase_unit).tolist(),
            purchases["unit_okpd_code"].tolist(),
        )
    ]
    seller_service = [has_service_signal(text) for text in (seller_name + " " + seller_category).tolist()]
    purchase_tokens = [significant_tokens(text) for text in purchase_text.tolist()]
    purchase_item_tokens = [significant_tokens(text) for text in purchase_item_text.tolist()]
    seller_tokens = [significant_tokens(text) for text in seller_text.tolist()]

    print(f"[3/6] Building hybrid retrieval index")
    (
        word_vectorizer,
        char_vectorizer,
        x_lot_docs,
        x_item_docs,
        svd,
        lot_embeddings,
        vectorizer_report,
    ) = build_or_load_hybrid_indices(
        root=root,
        purchases_path=purchases_path,
        lot_text=lot_text,
        purchase_item_text=purchase_item_text,
        args=args,
    )
    q_docs = transform_with_vectorizers(seller_text, word_vectorizer, char_vectorizer, args)
    q_embeddings = None
    if svd is not None and lot_embeddings is not None:
        q_embeddings = normalize_dense_rows(svd.transform(q_docs).astype(np.float32, copy=False))

    clf = None
    okpd_report = {"enabled": False, "reason": "disabled"}
    okpd_probabilities: np.ndarray | None = None
    if not args.no_okpd_classifier:
        print(f"[4/6] Training local OKPD classifier")
        clf, okpd_report = train_okpd_classifier(
            x_docs=x_item_docs,
            purchases=purchases,
            min_class_count=args.okpd_min_class_count,
            max_iter=args.okpd_max_iter,
        )
        if clf is not None:
            okpd_probabilities = clf.predict_proba(q_docs)
    else:
        print(f"[4/6] Skipping OKPD classifier")

    print(f"[5/6] Ranking candidates")
    rows: list[dict] = []
    item_rows: list[dict] = []
    product_rows: list[dict] = []
    lot_to_indices = purchases.groupby("pn_lot", sort=False).indices
    class_to_index = {str(cls): idx for idx, cls in enumerate(clf.classes_)} if clf is not None else {}

    for chunk_start in range(0, q_docs.shape[0], args.chunk_size):
        chunk_end = min(q_docs.shape[0], chunk_start + args.chunk_size)
        lot_scores = q_docs[chunk_start:chunk_end] @ x_lot_docs.T
        semantic_scores = None
        if q_embeddings is not None and lot_embeddings is not None:
            semantic_scores = (q_embeddings[chunk_start:chunk_end] @ lot_embeddings.T).astype(np.float32, copy=False)
        for local_idx, product_idx in enumerate(range(chunk_start, chunk_end)):
            query_row = lot_scores.getrow(local_idx)
            lexical_candidates = topk_sparse_row(query_row, args.top_rows)
            semantic_candidates = (
                topk_dense_scores(semantic_scores[local_idx], args.semantic_top_lots)
                if semantic_scores is not None
                else []
            )
            candidate_lots: dict[int, dict[str, float]] = {}
            for lot_idx, score in lexical_candidates:
                current = candidate_lots.setdefault(int(lot_idx), {"lexical_score": 0.0, "semantic_score": 0.0})
                current["lexical_score"] = max(current["lexical_score"], float(score))
            for lot_idx, score in semantic_candidates:
                current = candidate_lots.setdefault(int(lot_idx), {"lexical_score": 0.0, "semantic_score": 0.0})
                current["semantic_score"] = max(current["semantic_score"], float(score))
            per_lot: dict[str, dict] = {}
            query_tokens = seller_tokens[product_idx]
            query_is_service = seller_service[product_idx]
            query_type = str(seller_product_type[product_idx])
            product_okpd_probs = okpd_probabilities[product_idx] if okpd_probabilities is not None else None

            for lot_idx, candidate_scores in candidate_lots.items():
                lot_id = lot_ids[lot_idx]
                lot_indices = np.asarray(lot_to_indices.get(lot_id, []), dtype=np.int32)
                if lot_indices.size == 0:
                    continue
                item_similarity = (q_docs[product_idx] @ x_item_docs[lot_indices].T).toarray().ravel()
                if item_similarity.size == 0:
                    continue
                best_item_pos = int(item_similarity.argmax())
                purchase_idx = int(lot_indices[best_item_pos])
                lexical_score = float(candidate_scores["lexical_score"])
                semantic_score = float(candidate_scores["semantic_score"])
                doc_prefix = str(purchase_okpd_prefix[purchase_idx])
                doc_type = str(purchase_product_type[purchase_idx])
                item_presence_score = float(item_similarity[best_item_pos])
                retrieval_score = (1.0 - args.embedding_weight) * lexical_score + args.embedding_weight * semantic_score
                text_score = (
                    args.item_presence_weight * item_presence_score
                    + (1.0 - args.item_presence_weight) * retrieval_score
                )
                okpd_bonus = 0.0
                if product_okpd_probs is not None and doc_prefix in class_to_index:
                    okpd_bonus = args.okpd_bonus_weight * float(product_okpd_probs[class_to_index[doc_prefix]])

                overlap = 0.0
                if query_tokens:
                    overlap = len(query_tokens & purchase_tokens[purchase_idx]) / len(query_tokens)
                overlap_bonus = args.overlap_bonus_weight * min(overlap, 0.8)

                service_penalty = 0.0
                if not query_is_service and purchase_service[purchase_idx]:
                    service_penalty = args.service_penalty

                type_score, type_relation = type_adjustment(
                    seller_type=query_type,
                    purchase_type=doc_type,
                    exact_bonus=args.type_exact_bonus,
                    compatible_bonus=args.type_compatible_bonus,
                    mismatch_penalty=args.type_mismatch_penalty,
                )
                expected_type_signal = has_expected_type_signal(
                    seller_type=query_type,
                    purchase_type=doc_type,
                    purchase_text=purchase_type_signal_text[purchase_idx],
                )
                type_missing_penalty = 0.0 if expected_type_signal else args.type_missing_keyword_penalty

                final_score = text_score + okpd_bonus + overlap_bonus + type_score - service_penalty - type_missing_penalty
                current = per_lot.get(lot_id)
                if current is None or final_score > current["final_score"]:
                    matched = sorted(query_tokens & purchase_tokens[purchase_idx])
                    per_lot[lot_id] = {
                        "product_index": product_idx,
                        "seller_id": seller.at[product_idx, "id"],
                        "seller_category": seller.at[product_idx, "Категория"],
                        "seller_name": seller.at[product_idx, "Наименование"],
                        "pn_lot": lot_id,
                        "platform_brief": purchases.at[purchase_idx, "platform_brief"],
                        "publish_date": purchases.at[purchase_idx, "publish_date"],
                        "platform_number": purchases.at[purchase_idx, "platform_number"],
                        "lot_number": purchases.at[purchase_idx, "lot_number"],
                        "procedure_name": purchases.at[purchase_idx, "procedure_name"],
                        "lot_subject": purchases.at[purchase_idx, "lot_subject"],
                        "matched_unit_name": purchases.at[purchase_idx, "unit_name"],
                        "unit_okpd_code": purchases.at[purchase_idx, "unit_okpd_code"],
                        "seller_product_type": query_type,
                        "matched_product_type": doc_type,
                        "type_relation": type_relation,
                        "lexical_score": lexical_score,
                        "semantic_score": semantic_score,
                        "retrieval_score": retrieval_score,
                        "item_presence_score": item_presence_score,
                        "text_score": text_score,
                        "okpd_bonus": okpd_bonus,
                        "overlap_bonus": overlap_bonus,
                        "type_score": type_score,
                        "has_expected_type_signal": expected_type_signal,
                        "type_missing_penalty": type_missing_penalty,
                        "service_penalty": service_penalty,
                        "final_score": final_score,
                        "score_100": clipped_score_100(final_score),
                        "confidence": confidence_label(final_score),
                        "matched_terms": ", ".join(matched[:20]),
                        "row_index": purchase_idx,
                    }

            ranked = sorted(per_lot.values(), key=lambda item: item["final_score"], reverse=True)
            ranked = ranked[: args.top_lots]
            for rank, item in enumerate(ranked, start=1):
                item["lot_rank"] = rank
            if product_okpd_probs is not None and clf is not None:
                okpd_prediction = format_okpd_prediction(clf.classes_, product_okpd_probs)
            else:
                okpd_prediction = ""
            best_score = ranked[0]["final_score"] if ranked else 0.0
            if ranked:
                best_lot = ranked[0]
                lot_indices = list(lot_to_indices.get(best_lot["pn_lot"], []))
                if lot_indices:
                    lot_similarity = (q_docs[product_idx] @ x_item_docs[lot_indices].T).toarray().ravel()
                    scored_items: list[dict] = []
                    for item_pos, purchase_idx in enumerate(lot_indices):
                        lexical_score = float(lot_similarity[item_pos])
                        doc_prefix = str(purchase_okpd_prefix[purchase_idx])
                        doc_type = str(purchase_product_type[purchase_idx])
                        okpd_bonus = 0.0
                        if product_okpd_probs is not None and doc_prefix in class_to_index:
                            okpd_bonus = args.item_okpd_bonus_weight * float(
                                product_okpd_probs[class_to_index[doc_prefix]]
                            )

                        overlap = 0.0
                        if query_tokens:
                            overlap = len(query_tokens & purchase_item_tokens[purchase_idx]) / len(query_tokens)
                        overlap_bonus = args.overlap_bonus_weight * min(overlap, 0.8)

                        service_penalty = 0.0
                        if not query_is_service and purchase_service[purchase_idx]:
                            service_penalty = args.service_penalty

                        type_score, type_relation = type_adjustment(
                            seller_type=query_type,
                            purchase_type=doc_type,
                            exact_bonus=args.item_type_exact_bonus,
                            compatible_bonus=args.item_type_compatible_bonus,
                            mismatch_penalty=args.item_type_mismatch_penalty,
                        )
                        expected_type_signal = has_expected_type_signal(
                            seller_type=query_type,
                            purchase_type=doc_type,
                            purchase_text=purchase_type_signal_text[purchase_idx],
                        )
                        type_missing_penalty = 0.0 if expected_type_signal else args.item_type_missing_keyword_penalty
                        service_blocked = (not query_is_service) and bool(purchase_service[purchase_idx])

                        final_score = lexical_score + okpd_bonus + overlap_bonus + type_score - service_penalty - type_missing_penalty
                        unit_value = purchases.at[purchase_idx, "unit_name"]
                        unit_missing = pd.isna(unit_value) or not str(unit_value).strip()
                        if unit_missing and int(purchase_idx) == int(best_lot["row_index"]):
                            # Some marketplaces provide only lot-level rows. In that case the
                            # selected lot itself is the best available "item" explanation.
                            final_score = max(final_score, float(best_lot["final_score"]))
                        matched = sorted(query_tokens & purchase_item_tokens[purchase_idx])
                        scored_items.append(
                            {
                                "product_index": product_idx,
                                "seller_id": seller.at[product_idx, "id"],
                                "seller_category": seller.at[product_idx, "Категория"],
                                "seller_name": seller.at[product_idx, "Наименование"],
                                "pn_lot": best_lot["pn_lot"],
                                "platform_brief": purchases.at[purchase_idx, "platform_brief"],
                                "publish_date": purchases.at[purchase_idx, "publish_date"],
                                "platform_number": purchases.at[purchase_idx, "platform_number"],
                                "lot_number": purchases.at[purchase_idx, "lot_number"],
                                "procedure_name": purchases.at[purchase_idx, "procedure_name"],
                                "lot_subject": purchases.at[purchase_idx, "lot_subject"],
                                "unit_name": unit_value,
                                "unit_display_name": display_text(unit_value, purchases.at[purchase_idx, "lot_subject"]),
                                "is_lot_level_row": unit_missing,
                                "unit_okpd_code": purchases.at[purchase_idx, "unit_okpd_code"],
                                "seller_product_type": query_type,
                                "item_product_type": doc_type,
                                "item_type_relation": type_relation,
                                "item_lexical_score": lexical_score,
                                "item_okpd_bonus": okpd_bonus,
                                "item_overlap_bonus": overlap_bonus,
                                "item_type_score": type_score,
                                "item_has_expected_type_signal": expected_type_signal,
                                "item_type_missing_penalty": type_missing_penalty,
                                "item_service_penalty": service_penalty,
                                "item_service_blocked": service_blocked,
                                "item_final_score": final_score,
                                "item_score_100": clipped_score_100(final_score),
                                "is_best_item": int(purchase_idx) == int(best_lot["row_index"]),
                                "matched_terms": ", ".join(matched[:20]),
                                "row_index": int(purchase_idx),
                            }
                        )
                    scored_items.sort(key=lambda item: item["item_final_score"], reverse=True)
                    best_item_score = scored_items[0]["item_final_score"] if scored_items else 0.0
                    for item in scored_items:
                        item["is_suitable_item"] = is_suitable_item(
                            score=item["item_final_score"],
                            best_score=best_item_score,
                            threshold=args.item_suitable_threshold,
                            relative_threshold=args.item_relative_threshold,
                        ) and bool(item["item_has_expected_type_signal"]) and not bool(item["item_service_blocked"])
                    if args.max_best_lot_items > 0:
                        scored_items = scored_items[: args.max_best_lot_items]
                    for rank, item in enumerate(scored_items, start=1):
                        item["item_rank_in_best_lot"] = rank
                    item_rows.extend(scored_items)
            product_rows.append(
                {
                    "product_index": product_idx,
                    "seller_id": seller.at[product_idx, "id"],
                    "seller_category": seller.at[product_idx, "Категория"],
                    "seller_name": seller.at[product_idx, "Наименование"],
                    "seller_product_type": query_type,
                    "top_final_score": best_score,
                    "top_score_100": clipped_score_100(best_score),
                    "top_confidence": confidence_label(best_score),
                    "okpd_prediction": okpd_prediction,
                    "returned_lots": len(ranked),
                }
            )
            rows.extend(ranked)

    matches = pd.DataFrame(rows)
    best_lot_items = pd.DataFrame(item_rows)
    product_summary = pd.DataFrame(product_rows)
    best_matches = matches[matches["lot_rank"] == 1].copy() if not matches.empty else matches

    review_rows: list[dict] = []
    if not best_matches.empty:
        items_by_product = {
            int(product_idx): group.copy()
            for product_idx, group in best_lot_items.groupby("product_index", sort=False)
        }
        for match in best_matches.itertuples(index=False):
            product_idx = int(match.product_index)
            items = items_by_product.get(product_idx, pd.DataFrame())
            suitable = items[items["is_suitable_item"].astype(bool)] if not items.empty else items
            all_items_count = int(len(items))
            suitable_count = int(len(suitable))
            best_position = display_text(match.matched_unit_name, match.lot_subject)
            review_rows.append(
                {
                    "товар_селлера": match.seller_name,
                    "категория_селлера": match.seller_category,
                    "id_товара": match.seller_id,
                    "тип_товара_селлера": match.seller_product_type,
                    "предлагаемый_лот": match.pn_lot,
                    "предмет_лота": match.lot_subject,
                    "название_процедуры": match.procedure_name,
                    "лучшая_позиция_в_лоте": best_position,
                    "позиция_взята_из_предмета_лота": pd.isna(match.matched_unit_name)
                    or not str(match.matched_unit_name).strip(),
                    "тип_предложения": match.matched_product_type,
                    "тип_совпал": match.type_relation,
                    "окпд_лучшей_позиции": match.unit_okpd_code,
                    "релевантность_0_100": match.score_100,
                    "уверенность": match.confidence,
                    "совпавшие_слова": match.matched_terms,
                    "подходящих_позиций_в_лоте": suitable_count,
                    "всего_позиций_в_лоте": all_items_count,
                    "подходящие_позиции_списком": compact_join(
                        suitable["unit_display_name"], args.review_items_limit
                    )
                    if suitable_count
                    else "",
                    "площадка": match.platform_brief,
                    "дата_публикации": match.publish_date,
                    "номер_процедуры": match.platform_number,
                    "номер_лота": match.lot_number,
                }
            )
    review = pd.DataFrame(review_rows)
    suspicious = pd.DataFrame()
    probable_errors = pd.DataFrame()
    issue_audit = pd.DataFrame()
    issue_summary = pd.DataFrame()
    category_issue_summary = pd.DataFrame()
    category_metrics = pd.DataFrame()
    if not review.empty:
        repeated_lot_ids = set(review["предлагаемый_лот"].value_counts()[lambda s: s >= args.repeated_lot_threshold].index)
        suspicion_flags = review.apply(lambda row: build_suspicion_flags(row, repeated_lot_ids), axis=1)
        suspicious = review.assign(
            флаги_проверки=suspicion_flags.map("; ".join),
            количество_флагов=suspicion_flags.map(len),
        )
        suspicious = suspicious[suspicious["количество_флагов"] > 0].sort_values(
            ["количество_флагов", "релевантность_0_100"],
            ascending=[False, True],
        )
        probable_errors = suspicious[
            ~suspicious["флаги_проверки"].eq("один лот предложен многим товарам")
        ].copy()
        issue_audit, issue_summary, category_issue_summary = build_issue_outputs(review, suspicious)
        category_metrics = build_category_metrics(review, probable_errors)
    metrics = {}
    if not review.empty:
        products_count = max(1, len(review))
        no_suitable = review["подходящих_позиций_в_лоте"].astype(int).eq(0)
        seller_type_known = review["тип_товара_селлера"].fillna("").ne("")
        proposal_type_known = review["тип_предложения"].fillna("").ne("")
        type_known_pair = seller_type_known & proposal_type_known
        type_mismatch = review["тип_совпал"].eq("mismatch")
        service_flags = suspicious["флаги_проверки"].fillna("").str.contains("сервисный/ремонтный", regex=False)
        only_repeated = suspicious["флаги_проверки"].fillna("").eq("один лот предложен многим товарам")
        metrics = {
            "products": int(len(review)),
            "high_confidence": int(review["уверенность"].eq("high").sum()),
            "medium_or_high_confidence": int(review["уверенность"].isin(["medium", "high"]).sum()),
            "low_confidence": int(review["уверенность"].eq("low").sum()),
            "low_confidence_rate": float(review["уверенность"].eq("low").sum() / products_count),
            "products_without_suitable_item": int(no_suitable.sum()),
            "products_without_suitable_item_rate": float(no_suitable.sum() / products_count),
            "lot_level_explanations": int(review["позиция_взята_из_предмета_лота"].astype(bool).sum()),
            "lot_level_explanation_rate": float(review["позиция_взята_из_предмета_лота"].astype(bool).sum() / products_count),
            "seller_type_known": int(seller_type_known.sum()),
            "proposal_type_known": int(proposal_type_known.sum()),
            "type_known_pairs": int(type_known_pair.sum()),
            "type_exact": int(review["тип_совпал"].eq("exact").sum()),
            "type_compatible": int(review["тип_совпал"].eq("compatible").sum()),
            "type_mismatch": int(type_mismatch.sum()),
            "type_mismatch_rate_on_known_pairs": float(type_mismatch.sum() / max(1, type_known_pair.sum())),
            "suspicious_rows": int(len(suspicious)),
            "suspicious_only_repeated_lot": int(only_repeated.sum()) if not suspicious.empty else 0,
            "probable_error_rows": int(len(probable_errors)),
            "probable_error_rate": float(len(probable_errors) / products_count),
            "service_mismatch_rows": int(service_flags.sum()) if not suspicious.empty else 0,
        }

    print(f"[6/6] Writing outputs")
    matches_path = out_dir / "ml_baseline_matches.csv"
    best_lot_items_path = out_dir / "ml_baseline_best_lot_items.csv"
    product_summary_path = out_dir / "ml_baseline_product_summary.csv"
    review_path = out_dir / "ml_baseline_review.csv"
    suspicious_path = out_dir / "ml_baseline_suspicious.csv"
    probable_errors_path = out_dir / "ml_baseline_probable_errors.csv"
    category_metrics_path = out_dir / "ml_baseline_category_metrics.csv"
    issue_summary_path = out_dir / "ml_baseline_issue_summary.csv"
    issue_audit_path = out_dir / "ml_baseline_issue_audit.csv"
    category_issue_summary_path = out_dir / "ml_baseline_category_issue_summary.csv"
    metrics_path = out_dir / "ml_baseline_metrics.json"
    metrics_csv_path = out_dir / "ml_baseline_metrics.csv"
    run_summary_path = out_dir / "ml_baseline_run_summary.json"

    matches.to_csv(matches_path, index=False, encoding="utf-8-sig")
    best_lot_items.to_csv(best_lot_items_path, index=False, encoding="utf-8-sig")
    product_summary.to_csv(product_summary_path, index=False, encoding="utf-8-sig")
    review.to_csv(review_path, index=False, encoding="utf-8-sig")
    suspicious.to_csv(suspicious_path, index=False, encoding="utf-8-sig")
    probable_errors.to_csv(probable_errors_path, index=False, encoding="utf-8-sig")
    category_metrics.to_csv(category_metrics_path, index=False, encoding="utf-8-sig")
    issue_summary.to_csv(issue_summary_path, index=False, encoding="utf-8-sig")
    issue_audit.to_csv(issue_audit_path, index=False, encoding="utf-8-sig")
    category_issue_summary.to_csv(category_issue_summary_path, index=False, encoding="utf-8-sig")
    metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
    pd.DataFrame([{"metric": key, "value": value} for key, value in metrics.items()]).to_csv(
        metrics_csv_path, index=False, encoding="utf-8-sig"
    )

    confidence_counts = Counter(matches["confidence"]) if not matches.empty else Counter()
    run_summary = {
        "runtime_sec": round(time.time() - started_at, 2),
        "inputs": {
            "purchases": str(purchases_path),
            "seller_matrix": str(seller_path),
        },
        "outputs": {
            "matches": str(matches_path),
            "best_lot_items": str(best_lot_items_path),
            "product_summary": str(product_summary_path),
            "review": str(review_path),
            "suspicious": str(suspicious_path),
            "probable_errors": str(probable_errors_path),
            "category_metrics": str(category_metrics_path),
            "issue_summary": str(issue_summary_path),
            "issue_audit": str(issue_audit_path),
            "category_issue_summary": str(category_issue_summary_path),
            "metrics": str(metrics_path),
            "metrics_csv": str(metrics_csv_path),
        },
        "params": vars(args),
        "quality_report": quality_report,
        "seller_sampling": seller_sampling_report,
        "tfidf": {
            "lot_matrix_shape": list(x_lot_docs.shape),
            "lot_matrix_nnz": int(x_lot_docs.nnz),
            "item_matrix_nnz": int(x_item_docs.nnz),
            "query_matrix_shape": list(q_docs.shape),
            **vectorizer_report,
        },
        "okpd_classifier": okpd_report,
        "results": {
            "products": int(len(seller)),
            "match_rows": int(len(matches)),
            "best_lot_item_rows": int(len(best_lot_items)),
            "review_rows": int(len(review)),
            "suspicious_rows": int(len(suspicious)),
            "probable_error_rows": int(len(probable_errors)),
            "issue_rows": int(len(issue_audit)),
            "categories_covered": int(review["категория_селлера"].nunique()) if not review.empty else 0,
            "metrics": metrics,
            "confidence_counts": {key: int(val) for key, val in confidence_counts.items()},
            "products_high": int((product_summary["top_confidence"] == "high").sum()),
            "products_medium_or_high": int(product_summary["top_confidence"].isin(["medium", "high"]).sum()),
            "products_low": int((product_summary["top_confidence"] == "low").sum()),
            "top_score_median": float(product_summary["top_final_score"].median()),
            "top_score_p90": float(product_summary["top_final_score"].quantile(0.9)),
        },
    }
    run_summary_path.write_text(json.dumps(run_summary, ensure_ascii=False, indent=2), encoding="utf-8")

    print(json.dumps(run_summary["results"], ensure_ascii=False, indent=2))
    print(f"matches: {matches_path}")
    print(f"best_lot_items: {best_lot_items_path}")
    print(f"product_summary: {product_summary_path}")
    print(f"review: {review_path}")
    print(f"suspicious: {suspicious_path}")
    print(f"probable_errors: {probable_errors_path}")
    print(f"category_metrics: {category_metrics_path}")
    print(f"issue_summary: {issue_summary_path}")
    print(f"issue_audit: {issue_audit_path}")
    print(f"category_issue_summary: {category_issue_summary_path}")
    print(f"metrics: {metrics_path}")
    print(f"run_summary: {run_summary_path}")


## FunctionDef: parse_args


In [ ]:
def parse_args(argv: Iterable[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Local procurement matching ML baseline")
    parser.add_argument("--cwd", default=".", help="Workspace directory")
    parser.add_argument("--purchases", default="", help="Path to purchases CSV")
    parser.add_argument("--seller", default="", help="Path to seller matrix CSV")
    parser.add_argument("--output-dir", default="outputs", help="Output directory")
    parser.add_argument("--top-rows", type=int, default=80, help="Candidate lots per product from TF-IDF retrieval")
    parser.add_argument("--top-lots", type=int, default=3, help="Returned lots per seller product in priority order")
    parser.add_argument("--semantic-top-lots", type=int, default=40, help="Candidate lots per product from embedding retrieval")
    parser.add_argument("--chunk-size", type=int, default=32, help="Query chunk size for sparse scoring")
    parser.add_argument("--max-features", type=int, default=180_000, help="TF-IDF vocabulary cap")
    parser.add_argument("--min-df", type=int, default=2, help="TF-IDF min_df")
    parser.add_argument("--max-df", type=float, default=0.90, help="TF-IDF max_df")
    parser.add_argument(
        "--char-weight",
        type=float,
        default=0.18,
        help="Share of the retrieval score coming from character n-grams for unseen words and new categories",
    )
    parser.add_argument("--char-max-features", type=int, default=40_000, help="Character n-gram vocabulary cap")
    parser.add_argument("--char-min-df", type=int, default=3, help="Character n-gram min_df")
    parser.add_argument("--char-max-df", type=float, default=0.98, help="Character n-gram max_df")
    parser.add_argument("--char-ngram-min", type=int, default=3, help="Character n-gram lower bound")
    parser.add_argument("--char-ngram-max", type=int, default=5, help="Character n-gram upper bound")
    parser.add_argument("--embedding-dim", type=int, default=192, help="Dense embedding dimension built locally with TruncatedSVD; 0 disables embeddings")
    parser.add_argument("--embedding-weight", type=float, default=0.22, help="Share of lot retrieval score coming from dense embeddings")
    parser.add_argument("--embedding-random-state", type=int, default=42, help="Random state for TruncatedSVD embeddings")
    parser.add_argument("--index-cache-dir", default=".cache/ml_baseline", help="Directory for cached retrieval indices")
    parser.add_argument("--rebuild-index-cache", action="store_true", help="Force rebuilding cached TF-IDF and embedding indices")
    parser.add_argument("--lot-items-limit", type=int, default=12, help="How many unique item names to fold into each lot text")
    parser.add_argument("--lot-tags-limit", type=int, default=6, help="How many unique tags to fold into each lot text")
    parser.add_argument("--no-okpd-classifier", action="store_true", help="Disable local OKPD classifier")
    parser.add_argument("--okpd-min-class-count", type=int, default=20, help="Minimum rows per OKPD class")
    parser.add_argument("--okpd-max-iter", type=int, default=8, help="SGDClassifier max_iter")
    parser.add_argument("--okpd-bonus-weight", type=float, default=0.16, help="OKPD probability bonus weight")
    parser.add_argument(
        "--item-presence-weight",
        type=float,
        default=0.65,
        help="Share of row retrieval score that must come from the concrete lot item text, not only lot subject/procedure",
    )
    parser.add_argument("--type-exact-bonus", type=float, default=0.14, help="Bonus when seller and purchase product types match exactly")
    parser.add_argument(
        "--type-compatible-bonus",
        type=float,
        default=0.04,
        help="Small bonus when seller and purchase product types are in the same broad family",
    )
    parser.add_argument(
        "--type-mismatch-penalty",
        type=float,
        default=0.22,
        help="Penalty when known seller and purchase product types conflict",
    )
    parser.add_argument(
        "--type-missing-keyword-penalty",
        type=float,
        default=0.10,
        help="Penalty when seller type is known but the purchase row has no expected type signal",
    )
    parser.add_argument(
        "--item-okpd-bonus-weight",
        type=float,
        default=0.04,
        help="Smaller OKPD probability bonus used only for marking items inside the best lot",
    )
    parser.add_argument(
        "--item-type-exact-bonus",
        type=float,
        default=0.08,
        help="Item-level bonus when seller and purchase product types match exactly",
    )
    parser.add_argument(
        "--item-type-compatible-bonus",
        type=float,
        default=0.02,
        help="Item-level small bonus when seller and purchase product types are in the same broad family",
    )
    parser.add_argument(
        "--item-type-mismatch-penalty",
        type=float,
        default=0.18,
        help="Item-level penalty when known seller and purchase product types conflict",
    )
    parser.add_argument(
        "--item-type-missing-keyword-penalty",
        type=float,
        default=0.08,
        help="Item-level penalty when seller type is known but the lot item has no expected type signal",
    )
    parser.add_argument("--overlap-bonus-weight", type=float, default=0.08, help="Token overlap bonus weight")
    parser.add_argument("--service-penalty", type=float, default=0.24, help="Penalty for service lots on goods queries")
    parser.add_argument(
        "--item-suitable-threshold",
        type=float,
        default=0.16,
        help="Minimum item-level final score for marking an item as suitable inside the best lot",
    )
    parser.add_argument(
        "--item-relative-threshold",
        type=float,
        default=0.55,
        help="Minimum share of the best item score for marking an item as suitable inside the best lot",
    )
    parser.add_argument(
        "--max-best-lot-items",
        type=int,
        default=0,
        help="Maximum item rows to export per best lot; 0 means all rows",
    )
    parser.add_argument(
        "--review-items-limit",
        type=int,
        default=8,
        help="Maximum suitable item names to join into the compact review CSV",
    )
    parser.add_argument(
        "--repeated-lot-threshold",
        type=int,
        default=20,
        help="Deprecated: repeated lots are allowed and are no longer treated as suspicious",
    )
    parser.add_argument(
        "--per-category-limit",
        type=int,
        default=0,
        help="Take at most N seller rows per normalized category for fast category-balanced audits; 0 means use all rows",
    )
    return parser.parse_args(argv)

if __name__ == "__main__":
    run(parse_args())
